# 04 · Tracking inventory

**Question.** What does the game actually track, and where is the tracking unreliable?
**Answer.** 66 events, mostly in lower camelCase. Twelve quality issues, the main ones being purchases tracked on Android only, 10.4% of events without session id and a revenue property renamed mid-window. They are written up in `docs/tracking_plan.md`.
**Method.** Everything is observed in the data (`game_events.inventory`); the game's code and design documents were not available, so triggers and uses in the plan are interpretations. Example values are kept only for properties with 20 values or fewer, so that no identifier or receipt is published. Same output as `python -m game_events inventory`.

In [ ]:
from game_events import inventory
from game_events.data import connect_events

con, readable, unreadable = connect_events()
events, props, users = inventory.build_inventory(con)      # writes docs/*.csv; the property scan takes a few minutes
print(len(events), "event names,", len(props), "event/property pairs,", len(users), "player properties")

## 1. Events

In [ ]:
events.head(25)

Events that start after the first days or stop before the end point to tracking changes, or to the 2025-03-27 data gap:

In [ ]:
events[(events["first_day"].astype(str) > "2025-03-01") | (events["last_day"].astype(str) < "2025-04-09")]

## 2. Naming conventions

In [ ]:
events["style"] = events["event_name"].map(inventory.naming_style)
(events.groupby("style")["event_name"]
 .agg(count="count", examples=lambda s: ", ".join(s.head(8))))

Names with a digit (`gem1`, `gem2`, ...) are product names used as event names: each new product creates a new event, while `purchaseCompleted` already carries a product property.

## 3. Event properties

In [ ]:
props.head(25)

Properties missing from part of their event's rows, among the frequent events (optional properties, or added during the window):

In [ ]:
props[(props["pct_of_event"] < 100) & (props["n_events"] > 1000)].sort_values("pct_of_event").head(25)

## 4. Player properties

`user_properties_custom` is repeated on every event: a seeded 1% sample is enough to list its keys.

In [ ]:
users

## Output

`docs/event_inventory.csv`, `docs/event_properties.csv` and `docs/user_properties.csv`, the base of `docs/tracking_plan.md`. A property found on a small share of events cannot be used as a player attribute; very rare keys may be missing from the sample.